# 1. Environment & Configuration
Sets up the API security using `.env` and defines the core 100 tickers.

In [ ]:
import os
import time
import warnings
import pandas as pd
import numpy as np
from datetime import datetime
from dotenv import load_dotenv
from vnstock import Vnstock

warnings.filterwarnings("ignore")

# Securely load API Key from .env
load_dotenv()
if os.getenv("VNSTOCK_API_KEY"):
    Vnstock.api_key = os.getenv("VNSTOCK_API_KEY")

# ── Define Global 100 Tickers ──────────────────────────────────────────────
TICKERS_100 = [
    # 12 Core
    "VNM", "DHG", "FPT", "PNJ", "KDC", "ACB", "REE", "VIC", "PPC", "DPM", "AGF", "HPG",
    # Banking & Financials
    "VCB", "BID", "CTG", "MBB", "TCB", "VPB", "STB", "HDB", "SHB", "VIB", "TPB", "LPB", "MSB", "OCB", "SSB",
    "SSI", "HCM", "VND", "VCI", "SHS", "MBS", "FTS", "BSI", "CTS", "AGR",
    # Real Estate, Construction & Zones
    "VHM", "VRE", "NVL", "KDH", "NLG", "PDR", "DIG", "DXG", "KBC", "ITA", "SJS", "HDC", "HDG", "CTD", "HBC", "VCG", "IJC", "SZC", "IDC",
    # Materials & Industrials
    "HSG", "NKG", "HT1", "BCC", "DCM", "BFC", "DGC", "CSV", 
    # Retail & Consumer Goods
    "MSN", "SAB", "SBT", "MWG", "DGW", "PET", "FRT", 
    # Energy & Utilities
    "GAS", "PLX", "POW", "NT2", "VSH", "BWE", "TDM", "GEG",
    # Tech & Telecom
    "CMG", "ELC", "VGI",
    # Transport & Logistics
    "GMD", "VJC", "HVN", "PVT", "HAH", "VOS",
    # Seafood, Textiles, Rubber, Plastics
    "VHC", "ANV", "FMC", "TCM", "TNG", "GIL", "DRC", "CSM", "PHR", "DPR", "GVR", "BMP", "NTP", "PTB"
]
# Ensure uniqueness and exactly 100 elements
TICKERS_100 = list(dict.fromkeys(TICKERS_100))[:100]

print(f"Pipeline configured for {len(TICKERS_100)} unique tickers.")

# 2. Data Acquisition (Extraction)
Retrieves historical OHLCV data directly from `Vnstock` taking advantage of long back-history.

In [ ]:
OUTPUT_DIR = "data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
RAW_FILE = os.path.join(OUTPUT_DIR, "raw_data.csv")

TODAY = datetime.today().strftime("%Y-%m-%d")
SLEEP_SEC = 4.0
frames = []

print(f"Starting Extraction Process for {len(TICKERS_100)} tickers...")
for i, ticker in enumerate(TICKERS_100, 1):
    print(f"[{i:03d}/{len(TICKERS_100)}] Fetching {ticker:5s}...", end=" ")
    try:
        # start='2000-01-01' trick pulls all available history up to today automatically
        stock = Vnstock().stock(symbol=ticker, source="VCI")
        df = stock.quote.history(start="2000-01-01", end=TODAY, interval="1D")
        
        if df is not None and not df.empty:
            df = df.rename(columns={"time": "date"})
            df["date"] = pd.to_datetime(df["date"])
            df["ticker"] = ticker
            frames.append(df)
            print(f"✓ {len(df):>5,} sessions.")
        else:
            print("✗ Empty/Unlisted.")
    except Exception as e:
        print(f"✗ Error: {e}")
    time.sleep(SLEEP_SEC)

raw = pd.concat(frames, ignore_index=True)
raw = raw.sort_values(["ticker", "date"]).reset_index(drop=True)

# Generate Static Ticker ID Mapping
unique_tickers = raw["ticker"].unique()
ticker_to_id = {t: idx + 1 for idx, t in enumerate(unique_tickers)}
raw["ticker_id"] = raw["ticker"].map(ticker_to_id)

# Reorder columns so ticker_id is next to ticker
cols = list(raw.columns)
cols.remove("ticker_id")
idx = cols.index("ticker")
cols.insert(idx + 1, "ticker_id")
raw = raw[cols]

raw.to_csv(RAW_FILE, index=False)
print(f"\n✅ Acquisition Complete. Payload saved to: {RAW_FILE} ({len(raw):,} rows)")

# 3. Feature Engineering (Transformation)
Computes 13 critical technical indicators independently for each stock.

In [ ]:
def compute_rsi(series, window=14):
    delta = series.diff()
    gain, loss = delta.clip(lower=0), (-delta.clip(upper=0))
    avg_gain = gain.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

def compute_atr(high, low, close, window=14):
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1/window, min_periods=window, adjust=False).mean()

def compute_obv(close, volume):
    sign = np.sign(close.diff()).fillna(0)
    return (sign * volume).cumsum()

def process_features(df):
    c, o, h, lo, v = df["close"], df["open"], df["high"], df["low"], df["volume"].astype(float)
    hl = h - lo
    
    df["hl_range"]       = hl / c
    df["body_ratio"]     = (c - o).abs() / hl.replace(0, np.nan)
    df["close_position"] = (c - lo) / hl.replace(0, np.nan)
    df["return_1d"]      = c.pct_change(1)
    
    obv = compute_obv(c, v)
    obv_lag5 = obv.shift(5)
    df["obv_change"]     = (obv - obv_lag5) / obv_lag5.abs().replace(0, np.nan)
    df["return_5d"]      = c.pct_change(5)
    df["rsi"]            = compute_rsi(c, window=14)
    df["atr_ratio"]      = compute_atr(h, lo, c, window=14) / c
    
    ema3, ema10 = c.ewm(span=3, adjust=False).mean(), c.ewm(span=10, adjust=False).mean()
    df["ema_ratio"]      = ema3 / ema10 - 1
    
    bb_mid   = c.rolling(10).mean()
    bb_std   = c.rolling(10).std(ddof=0)
    bb_upper, bb_lower = bb_mid + 2 * bb_std, bb_mid - 2 * bb_std
    df["bb_pctb"]        = (c - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)
    
    df["volume_ratio"]   = v / v.rolling(10).mean().replace(0, np.nan)
    
    ret = c.pct_change(1)
    df["vol_trend"]      = ret.rolling(4).std() / ret.rolling(10).std().replace(0, np.nan)
    
    macd_line = c.ewm(span=5, adjust=False).mean() - c.ewm(span=10, adjust=False).mean()
    signal    = macd_line.ewm(span=5, adjust=False).mean()
    df["macd_hist"]      = macd_line - signal
    
    return df

print("Applying Technical Indicator Transformations...")
engineered_frames = []
for ticker, df in raw.groupby("ticker"):
    df = df.copy().reset_index(drop=True)
    engineered = process_features(df)
    engineered_frames.append(engineered)

dataset = pd.concat(engineered_frames, ignore_index=True)
print(f"✅ Feature Engineering Complete.")

# 4. Target Labeling & Cleaning
Shifts prices to create forward-looking labels, drops non-computable Null rows, and caps outliers (1st to 99th percentile).

In [ ]:
FEATURE_COLS = [
    "hl_range", "body_ratio", "close_position", "return_1d", "obv_change", 
    "return_5d", "rsi", "atr_ratio", "ema_ratio", "bb_pctb", "volume_ratio", 
    "vol_trend", "macd_hist"
]

# Generate 1D Label
labeled_frames = []
for ticker, grp in dataset.groupby("ticker"):
    grp = grp.copy().reset_index(drop=True)
    grp["label"] = (grp["close"].shift(-1) > grp["close"]).astype(int)
    grp = grp.iloc[:-1] # Drop the very last day (impossible to label tomorrow yet)
    labeled_frames.append(grp)
    
dataset = pd.concat(labeled_frames, ignore_index=True)

# Drop Nulls (from Indicator Warmup and calculation edge-cases)
before_len = len(dataset)
dataset = dataset.dropna(subset=FEATURE_COLS + ["label"]).reset_index(drop=True)
print(f"Dropped {before_len - len(dataset):,} null rows.")

# Statistical Outlier Clipping (Winsorize 1st and 99th percentiles)
for col in FEATURE_COLS:
    p01 = dataset[col].quantile(0.01)
    p99 = dataset[col].quantile(0.99)
    dataset[col] = dataset[col].clip(lower=p01, upper=p99)

print(f"✅ Cleaning Complete. Retained rows: {len(dataset):,}")

# 5. Global Export (Load)
Writes outputs to unified `total_stocks.csv` and disaggregates into `singular_stock/` directory.

In [ ]:
TOTAL_FILE = os.path.join(OUTPUT_DIR, "total_stocks.csv")
SINGULAR_DIR = os.path.join(OUTPUT_DIR, "singular_stock")
os.makedirs(SINGULAR_DIR, exist_ok=True)

# 1. Save globally
dataset.to_csv(TOTAL_FILE, index=False)

# 2. Save singularly
for ticker, grp in dataset.groupby("ticker"):
    out_path = os.path.join(SINGULAR_DIR, f"{ticker}.csv")
    grp.to_csv(out_path, index=False)

print(f"{'='*60}")
print(f"✅ PIPELINE FINISHED SUCCESSFULLY")
print(f"{'='*60}")
print(f"Total Tickers  : {dataset['ticker'].nunique()}")
print(f"Total Rows     : {len(dataset):,}")
print(f"Global File    : {TOTAL_FILE}")
print(f"Singular Files : {SINGULAR_DIR}/ (Contains {dataset['ticker'].nunique()} files)")